# tool_01 结构准备

## 功能
1. **结构转移**：扫描 ZPE 目录，将 .vasp/POSCAR 复制到任务目录
2. **原子固定**：固定基底 Cu 原子（Selective Dynamics F F F）
3. **Input 文件生成**：INCAR、KPOINTS、POTCAR

In [ ]:
import os
import shutil
import subprocess

In [ ]:
# ============================================
# 从 shared/config.py 读取（clio 可直接修改 config.py）
# ============================================
import sys
from pathlib import Path
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
cfg = load_config()
zpe_root = cfg.ZPE_ROOT

In [ ]:
# ============================
# INCAR / KPOINTS 模板（运行规则 2.4）
# ============================
INCAR_template = """ISTART = 1

ENCUT = 450
PREC = ACCURATE
LREAL = .FALSE.
ISMEAR = 1
SIGMA  = 0.2
GGA    = PE
NELM   = 120
NELMIN = 8
NELMDL = -5
EDIFF  = 1E-8
EDIFFG = -0.01
NSW    = 1
ISIF   = 2
IBRION = 5
POTIM  = 0.01
NFREE  = 2
NWRITE = 3

LWAVE  = .FALSE.
LCHARG = .FALSE.

ISPIN = 2
LORBIT = 11
IVDW = 12
IDIPOL = 3
LDIPOL = .TRUE.
NCORE = 8
ISYM = 0
IALGO = 48
"""

KPOINTS_template = """Automatic mesh
0
Gamma
4 4 1
0 0 0
"""

In [ ]:
def scan_zpe_structure(zpe_root):
    """扫描 ZPE 文件夹，返回 (结构文件路径列表, job目录路径列表)"""
    structure_files = []
    job_dirs = []
    has_standard = any(os.path.exists(os.path.join(zpe_root, b)) for b in ["1_body", "2_body"])
    
    if has_standard:
        for body_type in ["1_body", "2_body"]:
            body_path = os.path.join(zpe_root, body_type)
            if not os.path.exists(body_path):
                continue
            for molecule_dir in os.listdir(body_path):
                molecule_path = os.path.join(body_path, molecule_dir)
                if not os.path.isdir(molecule_path):
                    continue
                for file in os.listdir(molecule_path):
                    if file.endswith(".vasp"):
                        struct_path = os.path.join(molecule_path, file)
                        structure_files.append(struct_path)
                        basename = os.path.splitext(file)[0]
                        job_dirs.append(os.path.join(molecule_path, basename))
                    elif file == "POSCAR":
                        struct_path = os.path.join(molecule_path, file)
                        structure_files.append(struct_path)
                        job_dirs.append(os.path.join(molecule_path, "job_POSCAR"))
    else:
        for file in os.listdir(zpe_root):
            if file.endswith(".vasp"):
                struct_path = os.path.join(zpe_root, file)
                structure_files.append(struct_path)
                job_dirs.append(os.path.join(zpe_root, os.path.splitext(file)[0]))
            elif file == "POSCAR":
                structure_files.append(os.path.join(zpe_root, file))
                job_dirs.append(os.path.join(zpe_root, "job_POSCAR"))
    return structure_files, job_dirs


def fix_all_cu_atoms(poscar_path):
    """将 POSCAR 中 Cu 基底原子固定为 F F F。"""
    with open(poscar_path, 'r') as f:
        lines = f.readlines()
    if len(lines) < 8:
        return False
    if "Selective dynamics" not in ''.join(lines):
        return False
    element_line = lines[5].strip().split()
    num_line = lines[6].strip().split()
    if not element_line or element_line[0].upper() != 'CU':
        return False
    cu_count = int(num_line[0])
    coord_start = next(i for i, l in enumerate(lines) if "Selective dynamics" in l) + 2
    modified = False
    for i in range(coord_start, coord_start + cu_count):
        if i < len(lines):
            parts = lines[i].split()
            if len(parts) >= 6:
                lines[i] = f"{parts[0]:>20} {parts[1]:>20} {parts[2]:>20}   F   F   F\n"
                modified = True
    if modified:
        with open(poscar_path, 'w') as f:
            f.writelines(lines)
    return modified

In [ ]:
# 主程序：结构转移 + Input 生成 + 原子固定
print(f"📌 扫描 ZPE: {zpe_root}")
structure_files, job_dirs = scan_zpe_structure(zpe_root)

if not structure_files:
    print("❌ 未找到 .vasp 或 POSCAR")
else:
    print(f"找到 {len(structure_files)} 个结构")
    for sf, jd in zip(structure_files, job_dirs):
        print(f"  - {os.path.relpath(jd, zpe_root)}")
    print(f"\n{'='*60}\n")
    
    for struct_file, job_dir in zip(structure_files, job_dirs):
        rel = os.path.relpath(job_dir, zpe_root)
        if os.path.exists(job_dir) and os.path.exists(os.path.join(job_dir, "INCAR")):
            print(f"⏭ 跳过（已存在）: {rel}")
            continue
        print(f">>> {rel}")
        os.makedirs(job_dir, exist_ok=True)
        shutil.copy(struct_file, os.path.join(job_dir, "POSCAR"))
        with open(os.path.join(job_dir, "INCAR"), "w") as f:
            f.write(INCAR_template)
        with open(os.path.join(job_dir, "KPOINTS"), "w") as f:
            f.write(KPOINTS_template)
        try:
            subprocess.run("vaspkit -task 103", shell=True, cwd=job_dir, capture_output=True, timeout=30)
            print(f"  ✔ POTCAR")
        except (FileNotFoundError, subprocess.TimeoutExpired):
            print(f"  ⏭ 跳过 POTCAR")
        fix_all_cu_atoms(os.path.join(job_dir, "POSCAR"))
        print(f"  ✔ Cu 固定")
    print(f"\n✔ 完成。下一步：运行 tool_02 生成提交脚本")
